# Mimi-VC v2: Improved Transformer Voice Conversion

This notebook implements **5 key improvements** over the baseline Transformer:

| # | Problem | Solution |
|---|---------|----------|
| 1 | CB1-7 predicted independently | **Autoregressive across codebooks** — CB(i+1) conditioned on CB(i) |
| 2 | Source acoustic tokens ignored | **Source acoustic conditioning** via cross-attention |
| 3 | All codebooks weighted equally | **Exponential loss weighting** — CB1 weighted 4x more than CB7 |
| 4 | Fixed positional encoding | **RoPE** for better length generalization |
| 5 | Single-pass prediction | **Hybrid mode** — Transformer for CB1-3, retrieval for CB4-7 |

### Architecture: `AcousticTranslatorV2`
```
Source CB0 (semantic) ──> Semantic Encoder (4 layers)
Source CB1-7 (acoustic) ──> Acoustic Encoder (2 layers)
                                    ↓ cross-attention
                            Decoder (4 layers, autoregressive across CBs)
                                    ↓
                            CB1 → CB2 → ... → CB7 (sequential prediction)
```

> Based on: *Mimi-VC: Voice Conversion via Disentangled Neural Audio Codec Token Manipulation*
> — Q.V. Nguyen, H. Nguyen, L.D. Vu

In [ ]:
# ============================================================
# 1. INSTALL & IMPORTS
# ============================================================
!pip install -q transformers torch torchaudio faiss-cpu librosa soundfile matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
import faiss
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import os, time, random, base64, math
from collections import defaultdict
from transformers import MimiModel, AutoFeatureExtractor
from IPython.display import Audio as PlayAudio, display, HTML
from google.colab import files, output
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Loading Mimi...')
mimi = MimiModel.from_pretrained('kyutai/mimi').to(device).eval()
feat_ext = AutoFeatureExtractor.from_pretrained('kyutai/mimi')

SR = feat_ext.sampling_rate
N_CB = mimi.config.num_quantizers
CB_SIZE = mimi.config.codebook_size
CB_DIM = mimi.config.codebook_dim
FRAME_RATE = mimi.config.frame_rate
print(f'Mimi: SR={SR}, {N_CB}x{CB_SIZE}, dim={CB_DIM}, {FRAME_RATE}Hz')

In [ ]:
# ============================================================
# 2. CORE FUNCTIONS
# ============================================================

def load_audio(path, target_sr=SR):
    wav, orig_sr = librosa.load(path, sr=None, mono=True)
    if orig_sr != target_sr:
        wav = librosa.resample(wav, orig_sr=orig_sr, target_sr=target_sr)
    return wav.astype(np.float32)

def encode(wav_np):
    inputs = feat_ext(raw_audio=wav_np, sampling_rate=SR, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mimi.encode(inputs['input_values'], inputs.get('padding_mask'))
    return out.audio_codes.cpu()

def decode(codes):
    with torch.no_grad():
        out = mimi.decode(codes.to(device))
    return out.audio_values.cpu().squeeze().numpy()

def get_codebook_embeddings(cb_idx=0):
    q = mimi.quantizer
    if hasattr(q, 'semantic_residual_vector_quantizer'):
        n_sem = q.num_semantic_quantizers
        if cb_idx < n_sem:
            layer = q.semantic_residual_vector_quantizer.layers[cb_idx]
        else:
            layer = q.acoustic_residual_vector_quantizer.layers[cb_idx - n_sem]
    elif hasattr(q, 'layers'):
        layer = q.layers[cb_idx]
    else:
        raise RuntimeError(f'Unknown quantizer: {type(q).__name__}')
    for attr in ['codebook.embed', 'codebook.embed_sum', 'codebook.embedding.weight']:
        obj = layer
        try:
            for part in attr.split('.'):
                obj = getattr(obj, part)
            w = obj.detach().cpu().float()
            if w.dim() == 3: w = w.squeeze(0)
            if w.shape[0] >= CB_SIZE: return w
        except AttributeError:
            continue
    raise RuntimeError('Cannot find codebook embeddings')

cb0_emb = get_codebook_embeddings(0).numpy()
print(f'CB0 embeddings: {cb0_emb.shape}')

# Audio recorder
def record_audio(save_path, max_seconds=180, label="Audio"):
    rec_id = f"rec_{random.randint(10000,99999)}"
    display(HTML(f'''
<div style="border:1px solid #ccc;padding:15px;border-radius:8px;max-width:500px;margin:10px 0">
  <h4 style="margin-top:0">Record {label}</h4>
  <button id="{rec_id}_go" onclick="{rec_id}_start()" style="background:#4CAF50;color:#fff;padding:10px 24px;border:none;border-radius:4px;cursor:pointer;font-size:14px;margin-right:8px">Start</button>
  <button id="{rec_id}_no" onclick="{rec_id}_stop()" disabled style="background:#f44336;color:#fff;padding:10px 24px;border:none;border-radius:4px;cursor:pointer;font-size:14px">Stop</button>
  <p id="{rec_id}_s" style="margin:10px 0 5px;font-weight:bold">Click Start.</p>
  <p id="{rec_id}_t" style="font-family:monospace;font-size:18px;margin:0">0:00</p>
  <audio id="{rec_id}_a" controls style="display:none;margin-top:10px;width:100%"></audio>
</div>
<script>var {rec_id}_m,{rec_id}_c,{rec_id}_i,{rec_id}_n;async function {rec_id}_start(){{var s=await navigator.mediaDevices.getUserMedia({{audio:{{sampleRate:48000,channelCount:1,echoCancellation:!0,noiseSuppression:!0}}}});{rec_id}_c=[];{rec_id}_m=new MediaRecorder(s,{{mimeType:"audio/webm;codecs=opus"}});{rec_id}_m.ondataavailable=e=>{rec_id}_c.push(e.data);{rec_id}_m.onstop=()=>{{var b=new Blob({rec_id}_c,{{type:"audio/webm"}});document.getElementById("{rec_id}_a").src=URL.createObjectURL(b);document.getElementById("{rec_id}_a").style.display="block";document.getElementById("{rec_id}_s").textContent="Done!";var r=new FileReader;r.onload=()=>google.colab.kernel.invokeFunction("nb.{rec_id}",[r.result.split(",")[1]],{{}});r.readAsDataURL(b);s.getTracks().forEach(t=>t.stop())}};{rec_id}_n=0;{rec_id}_i=setInterval(()=>{{++{rec_id}_n;document.getElementById("{rec_id}_t").textContent=Math.floor({rec_id}_n/60)+":"+String({rec_id}_n%60).padStart(2,"0");if({rec_id}_n>={max_seconds}){rec_id}_stop()}},1e3);{rec_id}_m.start(1e3);document.getElementById("{rec_id}_go").disabled=!0;document.getElementById("{rec_id}_no").disabled=!1;document.getElementById("{rec_id}_s").textContent="Recording...";document.getElementById("{rec_id}_s").style.color="#f44336"}}function {rec_id}_stop(){{clearInterval({rec_id}_i);{rec_id}_m&&{rec_id}_m.state==="recording"&&{rec_id}_m.stop();document.getElementById("{rec_id}_go").disabled=!1;document.getElementById("{rec_id}_no").disabled=!0}}</script>
'''))
    result = {'done': False}
    webm = save_path.replace('.wav', '.webm')
    def cb(b64):
        with open(webm, 'wb') as f: f.write(base64.b64decode(b64))
        result['done'] = True
    output.register_callback(f'nb.{rec_id}', cb)
    print(f'Waiting for {label}...')
    while not result['done']: time.sleep(0.5)
    wav = load_audio(webm)
    sf.write(save_path, wav, SR)
    print(f'Recorded {len(wav)/SR:.1f}s')
    return wav

print('Ready.')

In [ ]:
# ============================================================
# 3. IMPROVED MODEL: AcousticTranslatorV2
# ============================================================
#
# Key improvements over V1:
#
# 1. AUTOREGRESSIVE ACROSS CODEBOOKS
#    V1: predict CB1..CB7 independently from semantic tokens
#    V2: predict CB1 first, then CB2 conditioned on CB1, etc.
#    Why: RVQ is residual — CB2 encodes residual AFTER CB1.
#         Predicting them independently ignores this structure.
#
# 2. SOURCE ACOUSTIC CONDITIONING
#    V1: only semantic tokens (CB0) as input
#    V2: encode source CB1-7 with a separate encoder,
#        then cross-attend from decoder to source acoustics
#    Why: source acoustics carry prosody, energy, timing info
#         that the model can transform rather than generate from scratch.
#
# 3. ROTARY POSITION EMBEDDINGS (RoPE)
#    V1: learned absolute positional embeddings (max_len=1024)
#    V2: RoPE applied in attention — generalizes to any length
#
# 4. CODEBOOK-LEVEL EMBEDDING
#    V2 adds a codebook index embedding so the decoder knows
#    which codebook it's currently predicting.
#

class RotaryEmbedding(nn.Module):
    """Rotary Position Embeddings (RoPE)."""
    def __init__(self, dim, max_len=4096):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self._build_cache(max_len)

    def _build_cache(self, max_len):
        t = torch.arange(max_len, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        self.register_buffer('cos_cached', freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer('sin_cached', freqs.sin()[None, None, :, :], persistent=False)

    def forward(self, x):
        T = x.shape[2]
        return self.cos_cached[:, :, :T, :], self.sin_cached[:, :, :T, :]


def apply_rope(q, k, cos, sin):
    """Apply rotary embeddings to query and key."""
    def rotate(x, cos, sin):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)
    return rotate(q, cos, sin), rotate(k, cos, sin)


class RoPETransformerLayer(nn.Module):
    """Transformer layer with RoPE and optional cross-attention."""
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1, has_cross_attn=False):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.has_cross_attn = has_cross_attn

        # Self-attention
        self.norm1 = nn.LayerNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)
        self.rope = RotaryEmbedding(self.head_dim)

        # Cross-attention (optional)
        if has_cross_attn:
            self.norm_cross = nn.LayerNorm(d_model)
            self.cq_proj = nn.Linear(d_model, d_model)
            self.ck_proj = nn.Linear(d_model, d_model)
            self.cv_proj = nn.Linear(d_model, d_model)
            self.co_proj = nn.Linear(d_model, d_model)

        # FFN
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)

    def _reshape(self, x, B, T):
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, cross_kv=None, causal=True):
        B, T, _ = x.shape

        # Self-attention with RoPE
        h = self.norm1(x)
        q = self._reshape(self.q_proj(h), B, T)
        k = self._reshape(self.k_proj(h), B, T)
        v = self._reshape(self.v_proj(h), B, T)

        cos, sin = self.rope(q)
        q, k = apply_rope(q, k, cos, sin)

        # Scaled dot-product attention
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if causal:
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            attn = attn.masked_fill(mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, self.d_model)
        x = x + self.dropout(self.o_proj(out))

        # Cross-attention to source acoustics
        if self.has_cross_attn and cross_kv is not None:
            h = self.norm_cross(x)
            T_kv = cross_kv.shape[1]
            cq = self._reshape(self.cq_proj(h), B, T)
            ck = self._reshape(self.ck_proj(cross_kv), B, T_kv)
            cv = self._reshape(self.cv_proj(cross_kv), B, T_kv)
            c_attn = (cq @ ck.transpose(-2, -1)) / math.sqrt(self.head_dim)
            c_attn = F.softmax(c_attn, dim=-1)
            c_out = (c_attn @ cv).transpose(1, 2).reshape(B, T, self.d_model)
            x = x + self.dropout(self.co_proj(c_out))

        # FFN
        x = x + self.ffn(self.norm2(x))
        return x


class AcousticTranslatorV2(nn.Module):
    """
    Improved Acoustic Token Translator.

    Changes from V1:
    - Autoregressive across codebooks (CB1 -> CB2 -> ... -> CB7)
    - Cross-attention to source acoustic tokens
    - RoPE positional encoding
    - Codebook index embedding
    """
    def __init__(self, n_vocab=2048, n_cb=7, d_model=256, n_heads=4,
                 n_enc_layers=4, n_src_acou_layers=2, n_dec_layers=4,
                 dropout=0.1):
        super().__init__()
        self.n_cb = n_cb
        self.d_model = d_model
        ff_dim = d_model * 4

        # --- Embeddings ---
        self.sem_embed = nn.Embedding(n_vocab, d_model)      # CB0 token embedding
        self.acou_embed = nn.Embedding(n_vocab, d_model)     # acoustic token embedding
        self.cb_embed = nn.Embedding(n_cb + 1, d_model)      # codebook index (0=semantic, 1-7=acoustic)
        self.src_acou_embeds = nn.ModuleList(                 # per-CB source acoustic embeddings
            [nn.Embedding(n_vocab, d_model) for _ in range(n_cb)]
        )

        # --- Semantic Encoder: encodes source CB0 ---
        self.sem_encoder = nn.ModuleList([
            RoPETransformerLayer(d_model, n_heads, ff_dim, dropout, has_cross_attn=False)
            for _ in range(n_enc_layers)
        ])
        self.sem_norm = nn.LayerNorm(d_model)

        # --- Source Acoustic Encoder: encodes source CB1-7 ---
        self.src_acou_encoder = nn.ModuleList([
            RoPETransformerLayer(d_model, n_heads, ff_dim, dropout, has_cross_attn=False)
            for _ in range(n_src_acou_layers)
        ])
        self.src_acou_norm = nn.LayerNorm(d_model)

        # --- Decoder: predicts target CB1-7 autoregressively ---
        self.decoder = nn.ModuleList([
            RoPETransformerLayer(d_model, n_heads, ff_dim, dropout, has_cross_attn=True)
            for _ in range(n_dec_layers)
        ])
        self.dec_norm = nn.LayerNorm(d_model)

        # --- Output head (shared across codebooks) ---
        self.output_head = nn.Linear(d_model, n_vocab)

    def encode_semantic(self, sem_tokens):
        """Encode semantic tokens. sem_tokens: [B, T]"""
        x = self.sem_embed(sem_tokens)
        for layer in self.sem_encoder:
            x = layer(x, causal=True)
        return self.sem_norm(x)

    def encode_source_acoustic(self, src_acou_tokens):
        """Encode source acoustic tokens. src_acou_tokens: [B, 7, T]"""
        B, n_cb, T = src_acou_tokens.shape
        # Sum embeddings across codebooks (like bag-of-codebooks per frame)
        x = torch.zeros(B, T, self.d_model, device=src_acou_tokens.device)
        for i in range(n_cb):
            x = x + self.src_acou_embeds[i](src_acou_tokens[:, i])
        x = x / n_cb
        for layer in self.src_acou_encoder:
            x = layer(x, causal=False)  # non-causal: can see full source
        return self.src_acou_norm(x)

    def forward(self, sem_tokens, src_acou_tokens, tgt_acou_tokens=None):
        """
        Training forward pass.

        Args:
            sem_tokens: [B, T] source semantic tokens (CB0)
            src_acou_tokens: [B, 7, T] source acoustic tokens (CB1-7)
            tgt_acou_tokens: [B, 7, T] target acoustic tokens (for teacher forcing)
        Returns:
            logits: [B, 7, T, V] predictions for each codebook
        """
        B, T = sem_tokens.shape

        # Encode source
        sem_enc = self.encode_semantic(sem_tokens)          # [B, T, D]
        src_acou_enc = self.encode_source_acoustic(src_acou_tokens)  # [B, T, D]

        # Autoregressive across codebooks with teacher forcing
        all_logits = []
        prev_cb_emb = torch.zeros(B, T, self.d_model, device=sem_tokens.device)

        for cb_i in range(self.n_cb):
            # Decoder input = semantic encoding + previous codebook embedding + CB index
            cb_idx = torch.full((B, T), cb_i + 1, dtype=torch.long, device=sem_tokens.device)
            dec_input = sem_enc + prev_cb_emb + self.cb_embed(cb_idx)

            # Cross-attend to source acoustics
            h = dec_input
            for layer in self.decoder:
                h = layer(h, cross_kv=src_acou_enc, causal=True)
            h = self.dec_norm(h)

            logits = self.output_head(h)  # [B, T, V]
            all_logits.append(logits)

            # Teacher forcing: use ground truth for next CB
            if tgt_acou_tokens is not None:
                prev_cb_emb = self.acou_embed(tgt_acou_tokens[:, cb_i])
            else:
                # Inference: use predicted
                prev_cb_emb = self.acou_embed(logits.argmax(dim=-1))

        return torch.stack(all_logits, dim=1)  # [B, 7, T, V]

    @torch.no_grad()
    def generate(self, sem_tokens, src_acou_tokens, temperature=0.8, top_k=50):
        """
        Inference: generate target acoustic tokens autoregressively across CBs.

        Returns: [B, 7, T] predicted acoustic tokens
        """
        B, T = sem_tokens.shape

        sem_enc = self.encode_semantic(sem_tokens)
        src_acou_enc = self.encode_source_acoustic(src_acou_tokens)

        predicted = []
        prev_cb_emb = torch.zeros(B, T, self.d_model, device=sem_tokens.device)

        for cb_i in range(self.n_cb):
            cb_idx = torch.full((B, T), cb_i + 1, dtype=torch.long, device=sem_tokens.device)
            dec_input = sem_enc + prev_cb_emb + self.cb_embed(cb_idx)

            h = dec_input
            for layer in self.decoder:
                h = layer(h, cross_kv=src_acou_enc, causal=True)
            h = self.dec_norm(h)

            logits = self.output_head(h) / temperature
            if top_k > 0:
                topk_v, _ = torch.topk(logits, top_k, dim=-1)
                logits[logits < topk_v[:, :, -1:]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            tokens = torch.multinomial(probs.reshape(-1, CB_SIZE), 1).reshape(B, T)
            predicted.append(tokens)

            # Feed predicted tokens to next codebook
            prev_cb_emb = self.acou_embed(tokens)

        return torch.stack(predicted, dim=1)  # [B, 7, T]


print('AcousticTranslatorV2 defined.')
print('Improvements: AR across CBs, source acoustic cross-attention, RoPE, CB embedding')

---
## Step 1: Record Target Speaker

In [ ]:
# ============================================================
# 4. RECORD TARGET SPEAKER
# ============================================================

target_wav = record_audio(
    save_path='/tmp/target_speaker.wav',
    max_seconds=300,
    label='Target Speaker (1-5 min recommended)'
)
target_wavs = [target_wav]
target_dur = len(target_wav) / SR
print(f'\nTarget: {target_dur:.1f}s')
display(PlayAudio(target_wav[:int(SR*15)], rate=SR))

In [ ]:
# ============================================================
# 4b. (ALT) Upload target audio
# ============================================================
# Skip if you recorded above.

uploaded = files.upload()
target_wavs = []
for f, d in uploaded.items():
    p = f'/tmp/tgt_{f}'
    with open(p, 'wb') as fh: fh.write(d)
    target_wavs.append(load_audio(p))
    print(f'{f}: {len(target_wavs[-1])/SR:.1f}s')
target_dur = sum(len(w) for w in target_wavs) / SR
print(f'Total: {target_dur:.1f}s')

---
## Step 2: Encode, Build Index & Train

In [ ]:
# ============================================================
# 5. ENCODE TARGET AUDIO
# ============================================================

CHUNK_SEC = 30
chunk_samples = int(CHUNK_SEC * SR)

all_chunks = []
for wav in target_wavs:
    for i in range(0, len(wav), chunk_samples):
        chunk = wav[i:i + chunk_samples]
        if len(chunk) > SR:
            all_chunks.append(chunk)

print(f'Encoding {len(all_chunks)} chunks...')
target_codes_list = []
for i, chunk in enumerate(all_chunks):
    codes = encode(chunk)
    target_codes_list.append(codes)
    print(f'  {i+1}/{len(all_chunks)}: {codes.shape[2]} frames')

all_target_codes = torch.cat(target_codes_list, dim=2)
T_total = all_target_codes.shape[2]
print(f'\nTotal: {T_total} frames ({T_total/FRAME_RATE:.1f}s)')

In [ ]:
# ============================================================
# 6. BUILD FAISS INDEX (for comparison & hybrid mode)
# ============================================================

all_sem = []
all_full = []
for t in range(T_total):
    all_sem.append(all_target_codes[0, 0, t].item())
    all_full.append(all_target_codes[0, :, t].numpy())

sem_arr = np.array(all_sem)
full_arr = np.stack(all_full)

tgt_emb = cb0_emb[sem_arr].astype(np.float32)
faiss.normalize_L2(tgt_emb)
faiss_index = faiss.IndexFlatIP(cb0_emb.shape[1])
faiss_index.add(tgt_emb)

def retrieval_vc(src_codes, k=8, alpha=0.5):
    T = src_codes.shape[2]
    s = src_codes[0, 0].numpy()
    e = cb0_emb[s].astype(np.float32)
    faiss.normalize_L2(e)
    d, idx = faiss_index.search(e, k)
    sel = np.zeros(T, dtype=np.int64)
    sel[0] = idx[0, 0]
    for t in range(1, T):
        c = idx[t]
        sel[t] = c[np.argmax(alpha*d[t] + (1-alpha)/(1+np.abs(c-sel[t-1]).astype(np.float32)))]
    out = src_codes.clone()
    for t in range(T):
        out[0, :, t] = torch.tensor(full_arr[sel[t]])
    return out

print(f'FAISS index: {faiss_index.ntotal} vectors')

In [ ]:
# ============================================================
# 7. TRAIN TRANSFORMER V2
# ============================================================

# --- Config ---
D_MODEL = 256
N_HEADS = 4
N_ENC_LAYERS = 4
N_SRC_ACOU_LAYERS = 2
N_DEC_LAYERS = 4
SEQ_LEN = 64
BATCH_SIZE = 8
LR = 3e-4

# Scale steps with data: more data -> more training
N_STEPS = min(int(500 * max(1, T_total / 500)), 3000)

# --- IMPROVEMENT 3: Exponential loss weighting ---
# CB1 matters ~4x more than CB7 in RVQ reconstruction
cb_weights = torch.tensor([2.0**(-(i)/2) for i in range(N_CB - 1)], device=device)
cb_weights = cb_weights / cb_weights.sum() * (N_CB - 1)  # normalize to sum=7

print(f'Training config:')
print(f'  Data: {T_total} frames ({T_total/FRAME_RATE:.1f}s)')
print(f'  Steps: {N_STEPS}, Batch: {BATCH_SIZE}, Seq: {SEQ_LEN}')
print(f'  Architecture: enc={N_ENC_LAYERS}L, src_acou={N_SRC_ACOU_LAYERS}L, dec={N_DEC_LAYERS}L')
print(f'  Loss weights: {[f"CB{i+1}={w:.2f}" for i, w in enumerate(cb_weights.cpu())]}')

# Prepare data
train_sem = all_target_codes[0, 0]      # [T]
train_acou = all_target_codes[0, 1:]     # [7, T]

# Initialize model
model_v2 = AcousticTranslatorV2(
    n_vocab=CB_SIZE, n_cb=N_CB-1, d_model=D_MODEL, n_heads=N_HEADS,
    n_enc_layers=N_ENC_LAYERS, n_src_acou_layers=N_SRC_ACOU_LAYERS,
    n_dec_layers=N_DEC_LAYERS
).to(device)
n_params = sum(p.numel() for p in model_v2.parameters())
print(f'  Params: {n_params/1e6:.1f}M\n')

opt = torch.optim.AdamW(model_v2.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, N_STEPS)

# --- Training ---
losses = []
model_v2.train()
t0 = time.time()

for step in range(N_STEPS):
    starts = torch.randint(0, max(1, T_total - SEQ_LEN), (BATCH_SIZE,))
    sem_b = torch.stack([train_sem[s:s+SEQ_LEN] for s in starts]).to(device)
    acou_b = torch.stack([train_acou[:, s:s+SEQ_LEN] for s in starts]).to(device)

    # Forward with teacher forcing
    # During training, source acoustic = target acoustic (self-reconstruction)
    logits = model_v2(sem_b, acou_b, tgt_acou_tokens=acou_b)  # [B, 7, T, V]

    # Weighted cross-entropy loss
    loss = 0
    for cb_i in range(N_CB - 1):
        ce = F.cross_entropy(logits[:, cb_i].reshape(-1, CB_SIZE), acou_b[:, cb_i].reshape(-1))
        loss = loss + cb_weights[cb_i] * ce
    loss = loss / (N_CB - 1)

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_v2.parameters(), 1.0)
    opt.step()
    sched.step()
    losses.append(loss.item())

    if (step + 1) % max(1, N_STEPS // 10) == 0:
        print(f'  Step {step+1:4d}/{N_STEPS} | Loss: {loss.item():.4f} | '
              f'LR: {sched.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s')

train_time = time.time() - t0
model_v2.eval()

# Plot
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(losses, color='#7030A0', linewidth=0.8)
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.set_title(f'V2 Training — {N_STEPS} steps, {train_time:.1f}s, {n_params/1e6:.1f}M params')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Final loss: {losses[-1]:.4f}')

---
## Step 3: Record Source & Convert

Converts with **3 methods** for comparison:
1. **FAISS+Smooth** — retrieval baseline
2. **Transformer V2** — improved learned approach
3. **Hybrid** — V2 for CB1-3 (important codebooks), retrieval for CB4-7 (fine detail)

In [ ]:
# ============================================================
# 8. RECORD SOURCE & CONVERT WITH ALL METHODS
# ============================================================

source_wav = record_audio(
    save_path='/tmp/source_audio.wav',
    max_seconds=120,
    label='Source Audio'
)
src_dur = len(source_wav) / SR
print(f'\nSource: {src_dur:.1f}s')

# Encode source
src_codes = encode(source_wav)
src_sem = src_codes[0, 0:1].to(device)      # [1, T]
src_acou = src_codes[0, 1:].unsqueeze(0).to(device)  # [1, 7, T]

# --- Method 1: FAISS+Smooth ---
print('\n1. FAISS+Smooth...')
t0 = time.time()
ret_codes = retrieval_vc(src_codes)
ret_wav = decode(ret_codes)
print(f'   {time.time()-t0:.2f}s')

# --- Method 2: Transformer V2 ---
print('2. Transformer V2...')
t0 = time.time()
torch.manual_seed(42)
pred_acou = model_v2.generate(src_sem, src_acou, temperature=0.8, top_k=50)
v2_codes = src_codes.clone()
v2_codes[0, 1:] = pred_acou[0].cpu()
v2_wav = decode(v2_codes)
print(f'   {time.time()-t0:.2f}s')

# --- Method 3: Hybrid (V2 for CB1-3, retrieval for CB4-7) ---
print('3. Hybrid (V2 + retrieval)...')
t0 = time.time()
hybrid_codes = src_codes.clone()
# Use Transformer for CB1-3 (most important acoustic codebooks)
hybrid_codes[0, 1:4] = pred_acou[0, :3].cpu()
# Use retrieval for CB4-7 (fine acoustic details)
hybrid_codes[0, 4:] = ret_codes[0, 4:]
hybrid_wav = decode(hybrid_codes)
print(f'   {time.time()-t0:.2f}s')

# --- Playback ---
print('\n' + '='*60)
print('RESULTS — Listen and compare')
print('='*60)

print('\n1. Original source:')
display(PlayAudio(source_wav, rate=SR))

print('\n2. FAISS+Smooth (retrieval, no training):')
display(PlayAudio(ret_wav, rate=SR))

print('\n3. Transformer V2 (learned, AR across CBs + source acoustic conditioning):')
display(PlayAudio(v2_wav, rate=SR))

print('\n4. Hybrid (V2 for CB1-3 + retrieval for CB4-7):')
display(PlayAudio(hybrid_wav, rate=SR))

print('\n5. Target speaker reference:')
display(PlayAudio(target_wavs[0][:int(SR*15)], rate=SR))

# Save
sf.write('/content/converted_retrieval.wav', ret_wav, SR)
sf.write('/content/converted_transformer_v2.wav', v2_wav, SR)
sf.write('/content/converted_hybrid.wav', hybrid_wav, SR)
print('\nSaved all 3 converted files.')

In [ ]:
# ============================================================
# 8b. (ALT) Upload source audio
# ============================================================

up = files.upload()
fname = list(up.keys())[0]
with open(f'/tmp/s_{fname}', 'wb') as f: f.write(up[fname])
source_wav = load_audio(f'/tmp/s_{fname}')
print(f'{fname}: {len(source_wav)/SR:.1f}s')

src_codes = encode(source_wav)
src_sem = src_codes[0, 0:1].to(device)
src_acou = src_codes[0, 1:].unsqueeze(0).to(device)

ret_codes = retrieval_vc(src_codes)
ret_wav = decode(ret_codes)

torch.manual_seed(42)
pred_acou = model_v2.generate(src_sem, src_acou, temperature=0.8)
v2_codes = src_codes.clone()
v2_codes[0, 1:] = pred_acou[0].cpu()
v2_wav = decode(v2_codes)

hybrid_codes = src_codes.clone()
hybrid_codes[0, 1:4] = pred_acou[0, :3].cpu()
hybrid_codes[0, 4:] = ret_codes[0, 4:]
hybrid_wav = decode(hybrid_codes)

for lbl, w in [('Original', source_wav), ('Retrieval', ret_wav),
               ('V2', v2_wav), ('Hybrid', hybrid_wav),
               ('Target ref', target_wavs[0][:int(SR*15)])]:
    print(f'\n{lbl}:'); display(PlayAudio(w, rate=SR))

sf.write('/content/converted_retrieval.wav', ret_wav, SR)
sf.write('/content/converted_transformer_v2.wav', v2_wav, SR)
sf.write('/content/converted_hybrid.wav', hybrid_wav, SR)

---
## Step 4: Visualize & Analyze

In [ ]:
# ============================================================
# 9. TOKEN HEATMAP COMPARISON
# ============================================================

tgt_vis = encode(target_wavs[0][:min(int(SR*10), len(target_wavs[0]))])
L = min(src_codes.shape[2], ret_codes.shape[2], v2_codes.shape[2],
        hybrid_codes.shape[2], tgt_vis.shape[2], 200)

fig, axes = plt.subplots(5, 1, figsize=(14, 12), constrained_layout=True)

items = [
    ('Source', src_codes),
    ('FAISS+Smooth', ret_codes),
    ('Transformer V2', v2_codes),
    ('Hybrid (V2+Retrieval)', hybrid_codes),
    ('Target Reference', tgt_vis),
]

for ax, (title, codes) in zip(axes, items):
    im = ax.imshow(codes[0, :, :L].numpy(), aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_ylabel('CB')
    ax.set_title(title, fontsize=11)
    ax.set_yticks(range(N_CB))
    ax.set_yticklabels([f'{i}' for i in range(N_CB)])
    plt.colorbar(im, ax=ax, shrink=0.5)

axes[-1].set_xlabel('Frame')
plt.suptitle('Token Comparison: 4 Methods', fontsize=13)
plt.savefig('/content/token_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# CB0 preservation
s = src_codes[0, 0, :L].numpy()
print(f'CB0 preservation (source content kept):')
for name, c in [('Retrieval', ret_codes), ('V2', v2_codes), ('Hybrid', hybrid_codes)]:
    match = np.mean(s == c[0, 0, :L].numpy()) * 100
    print(f'  {name:20s}: {match:5.1f}%')

In [ ]:
# ============================================================
# 10. PER-CODEBOOK ANALYSIS
# ============================================================
# Show which codebooks change the most between source and each method.

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

src_t = src_codes[0, :, :L].numpy()
tgt_t = tgt_vis[0, :, :L].numpy()

methods = [
    ('FAISS+Smooth', ret_codes[0, :, :L].numpy()),
    ('Transformer V2', v2_codes[0, :, :L].numpy()),
    ('Hybrid', hybrid_codes[0, :, :L].numpy()),
]

for ax, (name, conv_t) in zip(axes, methods):
    # Token change rate per codebook (vs source)
    change_vs_src = [np.mean(conv_t[i] != src_t[i]) * 100 for i in range(N_CB)]
    # Token match rate per codebook (vs target)
    match_vs_tgt = [np.mean(conv_t[i] == tgt_t[i]) * 100 for i in range(N_CB)]

    x = np.arange(N_CB)
    w = 0.35
    ax.bar(x - w/2, change_vs_src, w, label='Changed from source', color='#C55A11', alpha=0.8)
    ax.bar(x + w/2, match_vs_tgt, w, label='Matches target', color='#548235', alpha=0.8)
    ax.set_xlabel('Codebook')
    ax.set_ylabel('%')
    ax.set_title(name)
    ax.set_xticks(range(N_CB))
    ax.set_xticklabels([f'CB{i}' for i in range(N_CB)])
    ax.legend(fontsize=7)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 105)

plt.suptitle('Per-Codebook Token Change Analysis', fontsize=13)
plt.savefig('/content/codebook_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 11. TEMPERATURE SWEEP (V2)
# ============================================================

print('Transformer V2 temperature sweep:\n')

for temp in [0.2, 0.5, 0.8, 1.0, 1.5]:
    torch.manual_seed(42)
    pa = model_v2.generate(src_sem, src_acou, temperature=temp, top_k=50)
    tc = src_codes.clone()
    tc[0, 1:] = pa[0].cpu()
    tw = decode(tc)
    print(f'temp={temp}:')
    display(PlayAudio(tw, rate=SR))
    print()

In [ ]:
# ============================================================
# 12. DOWNLOAD
# ============================================================

print(f'Summary:')
print(f'  Target: {target_dur:.1f}s')
print(f'  Model: {n_params/1e6:.1f}M params, trained {train_time:.1f}s')
print(f'  Source: {src_dur:.1f}s\n')

files.download('/content/converted_retrieval.wav')
files.download('/content/converted_transformer_v2.wav')
files.download('/content/converted_hybrid.wav')
print('Downloaded all results.')

---
## (Optional) Convert More

In [ ]:
# ============================================================
# 13. RECORD & CONVERT MORE
# ============================================================

extra = record_audio(f'/tmp/extra_{random.randint(1,9999)}.wav', 120, 'More Audio')
ec = encode(extra)
es, ea = ec[0, 0:1].to(device), ec[0, 1:].unsqueeze(0).to(device)

rw = decode(retrieval_vc(ec))
torch.manual_seed(42)
pa = model_v2.generate(es, ea, temperature=0.8)
vc = ec.clone(); vc[0, 1:] = pa[0].cpu(); vw = decode(vc)
hc = ec.clone(); hc[0, 1:4] = pa[0, :3].cpu()
rc = retrieval_vc(ec); hc[0, 4:] = rc[0, 4:]; hw = decode(hc)

for l, w in [('Original', extra), ('Retrieval', rw), ('V2', vw), ('Hybrid', hw)]:
    print(f'\n{l}:'); display(PlayAudio(w, rate=SR))

eid = random.randint(1,9999)
for n, w in [('ret', rw), ('v2', vw), ('hyb', hw)]:
    sf.write(f'/content/extra_{n}_{eid}.wav', w, SR)
    files.download(f'/content/extra_{n}_{eid}.wav')